# Text-to-Text DPO 训练案例
核心是使用 HuggingFace Transformers 进行 text-to-text 任务的直接偏好优化（DPO）训练。
首先是加载必要的库

In [ ]:
import torch
import random
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW

加载一个预训练的文本生成模型（这里我采用的是gpt2，包括其模型和训练好的tokenizer），并将其移动到可用的计算设备（GPU 或 CPU）上

In [ ]:
model_name = "gpt2"  # 可以替换为任何text-to-text模型
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

这里定义并构建了一个简单的 text-to-text DPO 数据集，用于训练偏好优化模型

In [ ]:
class SimpleDpoDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=512):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        chosen_input = self.tokenizer(sample['prompt'] + sample['chosen'], truncation=True, max_length=self.max_length, return_tensors="pt")
        rejected_input = self.tokenizer(sample['prompt'] + sample['rejected'], truncation=True, max_length=self.max_length, return_tensors="pt")
        return {
            "chosen_input_ids": chosen_input["input_ids"].squeeze(0),
            "rejected_input_ids": rejected_input["input_ids"].squeeze(0),
        }

# 构造简单的样本
samples = [
    {"prompt": "Q: What is the capital of France?\\nA:", "chosen": " Paris", "rejected": " London"},
    {"prompt": "Q: What is 2+2?\\nA:", "chosen": " 4", "rejected": " 5"},
]

dataset = SimpleDpoDataset(samples, tokenizer)  # 创建数据集，加载样本和gpt2的tokenizer
dataloader = DataLoader(dataset, batch_size=1)  # 创建可迭代的数据流

这里定义了用于训练偏好优化模型的损失函数dpo_loss，使得相同的prompt下，模型输出用户更喜欢的答案，用chatgpt总结的markdown如下：本函数实现了 Direct Preference Optimization (DPO) 的核心损失计算逻辑，用于训练模型根据用户偏好输出更优质的回答。

In [ ]:
import torch.nn.functional as F

def dpo_loss(model, chosen_ids, rejected_ids):
    chosen_outputs = model(input_ids=chosen_ids, labels=chosen_ids)
    rejected_outputs = model(input_ids=rejected_ids, labels=rejected_ids)

    chosen_logps = -F.cross_entropy(chosen_outputs.logits[:, :-1, :].reshape(-1, chosen_outputs.logits.size(-1)),
                                    chosen_ids[:, 1:].reshape(-1), reduction='none')
    rejected_logps = -F.cross_entropy(rejected_outputs.logits[:, :-1, :].reshape(-1, rejected_outputs.logits.size(-1)),
                                      rejected_ids[:, 1:].reshape(-1), reduction='none')

    chosen_logps = chosen_logps.view(chosen_ids.size(0), -1).sum(dim=1)
    rejected_logps = rejected_logps.view(rejected_ids.size(0), -1).sum(dim=1)

    loss = -torch.log(torch.sigmoid(chosen_logps - rejected_logps)).mean()
    return loss

一个完整的 DPO 微调训练循环

In [17]:
optimizer = AdamW(model.parameters(), lr=1e-5)

model.train()
for epoch in range(3):
    for batch in dataloader:
        chosen_ids = batch["chosen_input_ids"].to(device)
        rejected_ids = batch["rejected_input_ids"].to(device)

        loss = dpo_loss(model, chosen_ids, rejected_ids)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Loss: {loss.item():.4f}")


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Loss: 0.0092
Loss: 0.1229
Loss: 3.2999
Loss: 0.0160
Loss: 0.2927
Loss: 0.2704


将微调后的模型和 tokenizer 保存

In [18]:
model.save_pretrained("dpo-text-model")
tokenizer.save_pretrained("dpo-text-model")


('dpo-text-model\\tokenizer_config.json',
 'dpo-text-model\\special_tokens_map.json',
 'dpo-text-model\\vocab.json',
 'dpo-text-model\\merges.txt',
 'dpo-text-model\\added_tokens.json',
 'dpo-text-model\\tokenizer.json')